# ARC-AGI-3 Closed-Loop Dual-Path ADL Agent

**Core loop:** `observe → compare two plans → act once → measure actual delta → ADL update → next move`.

The agent performs **AI Difference Learning after every real move**, not only at the end of a game. Each post-move update is restricted to the current game and current run, preserving the no-prior design.


In [1]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

taaf.kaggle: TRUE_SUBMISSION=False


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [2]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

0

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [3]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

taaf.kaggle: source bundle = /kaggle/input/datasets/jeroencottaar/taaf-kaggle-source-share
taaf.kaggle: input paths = {"driessmit1/arc3-vllm-h100-wheelhouse-v3": "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": "/kaggle/input/datasets/driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot", "jeroencottaar/taaf-kaggle-source-share": "/kaggle/input/datasets/jeroencottaar/taaf-kaggle-source-share"}


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands — installing wheels, fetching model weights, and so on.

In [4]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Bound generation before inference modules are imported. The original run
# allowed unlimited tool steps/output, amplifying stalls under high fan-out.
_score_runtime_env = {
    'LOCAL_ANALYZER_MAX_OUTPUT': os.environ.get('TAAF_MAX_OUTPUT_TOKENS', '8192'),
    'LOCAL_ANALYZER_TOOL_STEPS': os.environ.get('TAAF_TOOL_STEPS', '8'),
    'LOCAL_ANALYZER_TEMPERATURE': os.environ.get('TAAF_TEMPERATURE', '0.6'),
    'LOCAL_ANALYZER_TOP_P': os.environ.get('TAAF_TOP_P', '0.95'),
}
os.environ.update(_score_runtime_env)
_persisted_setup_env = json.loads(SETUP_ENV_PATH.read_text())
_persisted_setup_env.update(_score_runtime_env)
SETUP_ENV_PATH.write_text(json.dumps(_persisted_setup_env, indent=2, sort_keys=True) + '\n')
print(f'taaf.kaggle: bounded analyzer controls = {_score_runtime_env}')


taaf.kaggle: wrote /usr/local/lib/python3.12/dist-packages/taaf_kaggle_sources.pth (3 source roots)
taaf.kaggle: setup command: "$PYTHON" - <<'PYSETUP'
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

WHEELHOUSE_OWNER = 'driessmit1'
WHEELHOUSE_SLUG = 'arc3-vllm-h100-wheelhouse-v3'
MODEL_OWNER = 'driessmit1'
MODEL_SLUG = 'vrfai-qwen3-6-27b-fp8-hf-snapshot'
SERVED_MODEL_NAME = 'vrfai/Qwen3.6-27B-FP8'
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 1234
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'
VLLM_MAX_MODEL_LEN = 65536
ANALYZER_CONTEXT_WINDOW = 32768
VLLM_TENSOR_PARALLEL_SIZE = 1
WORKING_DIR = Path(os.environ['TAAF_KAGGLE_WORKING_DIR'])
SITE_PACKAGES = WORKING_DIR / 'vllm-site-packages'
VLLM_SERVER_LOG = WORKING_DIR / 'vllm-openai-server.log'
VLLM_SERVER_PID = WORKING_DIR / 'vllm-openai-server.pid'
INSTALL_STAMP = SITE_PACKAGES / f'.{WHEELHOUSE_SLUG}'
STAMP_TEXT = 'vllm==0.19.0 torch==2.10.0 flashinfer==0.6.6\n'


## 4.1 Minimal Duck Harness compatibility

Keep the bundled solver unchanged except for the missing neutral `ACTION7` reverse mapping. No prompt, score, environment, or execution method is patched.


In [5]:
import inference.agent.action_names as action_names
import inference.framework.solver as solver_module

action_names.MODEL_TO_ENGINE_ACTION["ACTION7"] = "ACTION7"
assert action_names.to_model_action("ACTION7") == "ACTION7"
assert action_names.to_engine_action("ACTION7") == "ACTION7"

PATCH_STATUS = {
    "patch": "minimal-action7-reverse-map-v1",
    "action7_reverse_mapping": True,
    "system_prompt_changed": False,
    "solver_methods_changed": False,
    "dataset_modified": False,
}
print(f"taaf.kaggle: compatibility={PATCH_STATUS}")


taaf.kaggle: compatibility={'patch': 'minimal-action7-reverse-map-v1', 'action7_reverse_mapping': True, 'system_prompt_changed': False, 'solver_methods_changed': False, 'dataset_modified': False}


## 4.2 Hard no-prior runtime contract

Declare and verify the information boundary before loading the benchmark. Only current-game observations, actions, rewards, transitions, Qwen's pass-1 transcript, and Gemma's review of that transcript may cross into pass 2.


In [6]:
NO_PRIOR_CONTRACT = {
    "allowed": [
        "same_current_run_same_game_observations",
        "same_current_run_same_game_actions",
        "same_current_run_same_game_rewards",
        "same_current_run_same_game_transitions",
        "same_current_run_same_game_post_move_adl",
        "same_current_run_same_game_dual_path_decisions",
    ],
    "forbidden": [
        "historical_transcripts",
        "yesterday_transcripts",
        "routebooks",
        "replays",
        "solved_paths",
        "hidden_labels",
        "cross_run_state",
        "cross_game_state",
        "prior_submission_state",
    ],
}
assert set(NO_PRIOR_CONTRACT["allowed"]).isdisjoint(
    NO_PRIOR_CONTRACT["forbidden"]
)
print("taaf.kaggle: strict current-game/current-run no-prior contract active")


taaf.kaggle: strict current-game/current-run no-prior contract active


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [7]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Fixed real-run configuration

Run every discovered game with concurrency 4 and strict no-prior behavior. A dynamic per-game cap keeps both complete passes inside Kaggle's nine-hour GPU runtime limit. No environment can be omitted.


In [8]:
STRICT_NO_PRIOR = True
TARGET_CONCURRENCY = 4

_original_game_budget = float(
    getattr(bm.solver, "max_runtime_s_per_game", 0.0) or 0.0
)
_original_concurrency = max(
    1, int(getattr(bm.solver, "concurrency", 1) or 1)
)
bm.solver.concurrency = TARGET_CONCURRENCY

# Optional grafts are constrained to the same current game and current run.
# Banking and transfer are intentionally never installed.
try:
    from taaf_grafts.composite import install as _install_taaf_grafts
except ModuleNotFoundError:
    _install_taaf_grafts = None

_graft_flags = {
    "shortcircuit": True,
    "efficiency": True,
    "retry_guard": True,
    "recovery": True,
    "context_window": int(
        os.environ.get("TAAF_CONTEXT_WINDOW", "32768")
    ),
}
if _install_taaf_grafts is not None:
    _install_taaf_grafts(bm, _graft_flags, expected_version=1)

assert bm.solver.concurrency == TARGET_CONCURRENCY
assert "banking" not in _graft_flags
assert "transfer" not in _graft_flags
print(
    "REAL RUN CONFIG: "
    f"strict_no_prior={STRICT_NO_PRIOR} "
    f"concurrency={TARGET_CONCURRENCY} "
    "games_required=all_discovered "
    f"source_per_game_budget={_original_game_budget}"
)


REAL RUN CONFIG: strict_no_prior=True concurrency=4 games_required=all_discovered source_per_game_budget=7920.0


## 7. Dual-Path ADL policy

The following cell replaces the normal analyzer factory with a `DualPathToolAgent`.
Every real action still travels through the ordinary Duck Harness action/tool mechanism.
The difference is that the analyzer is explicitly required to construct and compare
an EXPLOIT and EXPLORE candidate before making its single tool call.


In [9]:
# === CLOSED-LOOP DUAL-PATH ADL — BEFORE AND AFTER EVERY MOVE ===
import json
import os
from pathlib import Path
from inference.agent.tool_agent import ToolAgent

DUAL_PATH_ENABLED = True
POST_MOVE_ADL_ENABLED = True
DUAL_PATH_CANDIDATES = 2

DUAL_PATH_POLICY_LOG = WORKING_DIR / "dual_path_policy_events.jsonl"
POST_MOVE_ADL_LOG = WORKING_DIR / "post_move_adl_events.jsonl"

CLOSED_LOOP_ADL_INSTRUCTION = r"""
CLOSED-LOOP DUAL-PATH ADL POLICY — REQUIRED FOR EVERY REAL MOVE

You have one real environment trajectory. Never initialize, reset, fork, or
speculatively step a second environment.

Use only evidence from this CURRENT GAME and CURRENT RUN.

============================================================
PHASE 1 — BEFORE EVERY REAL ACTION: DUAL-PATH DIFFERENCE LEARNING
============================================================

Before issuing an environment action/tool call, construct exactly two candidate
actions from the SAME current observation and SAME current-game history.

Candidate A — EXPLOIT
- shortest legal move supported by confirmed evidence;
- maximize predicted immediate/near-term progress;
- avoid already observed failures and loops.

Candidate B — EXPLORE
- legal move with highest expected information gain;
- prefer actions likely to reveal mechanics, clickable objects, transitions,
  hidden controls, or meaningful frame changes;
- avoid already exhausted probes.

For both candidates estimate:
- legality
- predicted progress
- predicted frame change
- information gain
- loop risk
- action cost
- consistency with current-game evidence

Compare them using:
legality=.20
predicted_progress=.25
predicted_frame_change=.15
information_gain=.15
loop_avoidance=.10
action_efficiency=.10
current_game_consistency=.05

Before the real tool call, record:

DUAL_PATH_DECISION:
STEP=<integer>
A_ACTION=<candidate A>
A_PREDICTION=<expected effect>
B_ACTION=<candidate B>
B_PREDICTION=<expected effect>
SELECT=<A or B>
WHY=<short evidence-based reason>

Then issue exactly ONE real environment action: the selected action.

============================================================
PHASE 2 — IMMEDIATELY AFTER EVERY REAL ACTION: ADL UPDATE
============================================================

As soon as the environment/tool result for that action is returned, and BEFORE
planning the next move, compare the actual post-move state with the pre-move
state and with the selected candidate's prediction.

Record:

POST_MOVE_ADL:
STEP=<same integer>
ACTION=<actual committed action>
STATE_CHANGED=<yes/no/uncertain>
SCORE_DELTA=<observed score/reward delta if exposed, otherwise unknown>
LEVEL_DELTA=<observed level/progress delta if exposed, otherwise unknown>
PREDICTION_MATCH=<yes/partial/no/uncertain>
INFORMATION_GAIN=<0.0..1.0>
PROGRESS_VALUE=<-1.0..1.0>
LOOP_SIGNAL=<yes/no>
NOVEL_TRANSITION=<yes/no/uncertain>
LESSON=<compact current-game-only lesson>
NEXT_BIAS=<exploit/explore/neutral>

Interpretation:
- positive progress or newly revealed mechanics => reinforce the causal/action
  pattern for THIS game;
- unchanged/repeated state => downweight repeating that action in equivalent
  states;
- regression/game-over => strongly downweight that local pattern;
- surprising useful transition => increase exploration value of similar
  untested actions;
- prediction failure => reduce confidence in the assumption that produced it.

The POST_MOVE_ADL record becomes part of the CURRENT GAME'S working memory and
MUST influence the very next A/B comparison.

Do not wait until the end of a level or game. Perform POST_MOVE_ADL after EVERY
real move.

============================================================
STRICT NO-PRIOR BOUNDARY
============================================================

Allowed:
- observations, actions, rewards, transitions, hypotheses, and POST_MOVE_ADL
  records generated in this current run of this current game.

Forbidden:
- prior games
- prior submissions
- stored winning routes
- external solution memory
- yesterday/historical transcripts
- hidden labels
- replay libraries
- cross-game ADL memory

At a new game, begin with an empty game-specific ADL memory.
""".strip()


class ClosedLoopADLToolAgent(ToolAgent):
    """Duck ToolAgent with mandatory A/B pre-action reasoning and post-move ADL."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._system_prompt += "\n\n" + CLOSED_LOOP_ADL_INSTRUCTION


def _closed_loop_adl_analyzer_factory(game, index):
    model = (
        os.environ.get("INFERENCE_ANALYZER_MODEL")
        or os.environ.get("LOCAL_ANALYZER_MODEL_ID")
        or "vrfai/Qwen3.6-27B-FP8"
    )
    base_url = (
        os.environ.get("LOCAL_ANALYZER_BASE_URL")
        or os.environ.get("OPENAI_BASE_URL")
        or "http://127.0.0.1:1234/v1"
    )
    return ClosedLoopADLToolAgent(
        model=model,
        timeout=bm.solver.analyzer_timeout,
        save_request_logs=bm.solver.save_request_logs,
        base_url=base_url,
        provider="vllm",
    )


bm.solver.analyzer_factory = _closed_loop_adl_analyzer_factory

print("CLOSED-LOOP ADL ACTIVE", flush=True)
print("BEFORE MOVE: 2 internal plans -> select 1", flush=True)
print("AFTER MOVE: compare predicted vs actual delta -> update next decision", flush=True)
print("ENVIRONMENT PASSES PER GAME: 1", flush=True)
print("CROSS-GAME MEMORY: DISABLED", flush=True)


CLOSED-LOOP ADL ACTIVE
BEFORE MOVE: 2 internal plans -> select 1
AFTER MOVE: compare predicted vs actual delta -> update next decision
ENVIRONMENT PASSES PER GAME: 1
CROSS-GAME MEMORY: DISABLED


## 8. Run exactly one real environment trajectory per game

Competition and local modes share the same policy. Competition mode discovers games
from the official gateway. Local mode uses the mounted public `environment_files`.


In [10]:
# === ONE-ENVIRONMENT COMPETITION/LOCAL EXECUTION ===
import re
from urllib.request import urlopen

EXPECTED_LOCAL_GAME_COUNT = 25


def _game_key(value):
    text = str(value or "").strip()
    match = re.match(r"([A-Za-z0-9]+)", text)
    if not match:
        raise ValueError(f"Cannot derive game key from {value!r}")
    return match.group(1)


def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed no games.")
    if len(set(game_ids)) != len(game_ids):
        raise RuntimeError("Competition Arcade exposed duplicate game IDs.")
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _offline_games():
    import arc_agi
    import taaf.game_api

    root = Path(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
    )
    if not root.is_dir():
        raise FileNotFoundError(f"Local environment root missing: {root}")

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=str(root),
    )
    game_ids = [info.game_id for info in arcade.available_environments]
    if len(game_ids) != EXPECTED_LOCAL_GAME_COUNT:
        raise RuntimeError(
            f"Expected {EXPECTED_LOCAL_GAME_COUNT} local games; found {len(game_ids)}"
        )
    return [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec)
        for game_id in game_ids
    ]


def _wait_for_gateway(base_url, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Competition gateway did not become ready: {last_error}")


def _run_score(run):
    return float(getattr(run, "final_score", None) or 0.0)


def _run_levels(run):
    return int(getattr(run, "levels_completed", 0) or 0)


def _run_actions(run):
    return len(getattr(run, "history", ()) or ())


def _won(run):
    state = getattr(run, "state", "")
    return str(getattr(state, "name", state)).lower().endswith("won")


if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    game_apis = _competition_games()
    print(
        f"OFFICIAL COMPETITION MODE: {len(game_apis)} games, "
        "one environment trajectory per game",
        flush=True,
    )
else:
    game_apis = _offline_games()
    print(
        f"LOCAL MODE: {len(game_apis)} games, one environment trajectory per game",
        flush=True,
    )

RUN_GAME_COUNT = len(game_apis)
if RUN_GAME_COUNT < 1:
    raise RuntimeError("No ARC-AGI-3 games discovered.")

# One pass only. No hidden-game restart and no environment best-of-two.
bm.games = game_apis
bm.n_passes = 1
bm.game_weights = None
bm.solver.concurrency = TARGET_CONCURRENCY

# Runtime budget: reserve setup/teardown time, then divide the remaining time
# across the one legal environment pass.
NOTEBOOK_RUNTIME_TARGET_SECONDS = 8 * 60 * 60 + 30 * 60
NON_ENVIRONMENT_RESERVE_SECONDS = 70 * 60
available_environment_seconds = (
    NOTEBOOK_RUNTIME_TARGET_SECONDS - NON_ENVIRONMENT_RESERVE_SECONDS
)
runtime_safe_per_game = max(
    45.0,
    available_environment_seconds * TARGET_CONCURRENCY / RUN_GAME_COUNT,
)
RUN_PER_GAME_SECONDS = min(
    1500.0,
    runtime_safe_per_game,
    _original_game_budget if _original_game_budget > 0 else runtime_safe_per_game,
)
bm.solver.max_runtime_s_per_game = RUN_PER_GAME_SECONDS

run_manifest = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": RUN_GAME_COUNT,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "concurrency": TARGET_CONCURRENCY,
    "per_game_seconds": RUN_PER_GAME_SECONDS,
    "strict_no_prior": True,
    "second_environment_pass": False,
}

(WORKING_DIR / "dual_path_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "DUAL-PATH RUN START "
    f"games={RUN_GAME_COUNT} concurrency={TARGET_CONCURRENCY} "
    f"per_game_seconds={RUN_PER_GAME_SECONDS:.1f}",
    flush=True,
)

try:
    await bm.run(
        soft_end_time=None,
        runtime_environment=target,
        minimal_diagnostics=bool(TRUE_SUBMISSION),
    )
finally:
    # Keep the source bundle's own teardown behavior.
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print("Running teardown:", command, flush=True)
        subprocess.run(
            command,
            shell=True,
            check=False,
            cwd=WORKING_DIR,
            env=_command_env(),
        )

if len(getattr(bm, "game_runs", []) or []) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Expected {RUN_GAME_COUNT} completed game runs; "
        f"found {len(getattr(bm, 'game_runs', []) or [])}"
    )

for run in bm.game_runs:
    print(
        "DUAL-PATH SCORE "
        f"game={_game_key(run.game_id)} "
        f"score={_run_score(run):.6f} "
        f"levels={_run_levels(run)} "
        f"actions={_run_actions(run)}",
        flush=True,
    )


LOCAL MODE: 25 games, one environment trajectory per game
DUAL-PATH RUN START games=25 concurrency=4 per_game_seconds=1500.0
benchmark: regenerated diagnostics in /kaggle/working in 2.41s
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-08-18 10:24:22
ended:     in progress
mean score:    0.01
median score:  0.00
total actions: 295
total tokens:  62913
generated tokens/sec: 104.30 (job wallclock)
total wallclock: 2314.4s

per-game (mean across passes):
  ar25-0c556536: score=0.00, levels=0.0/8, actions=0, tokens=0
  bp35-0a0ad940: score=0.36, levels=1.0/9, actions=78, tokens=15358
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=0, tokens=0
  cn04-2fe56bfb: score=0.00, levels=0.0/6, actions=0, tokens=0
  dc22-fdcac232: score=0.00, levels=0.0/6, actions=0, tokens=0
  ft09-0d8bbf25: score=0.00, levels=0.0/6, actions=0, tokens=0
  g50t-5849a774: score=0.00, levels=0.0/7, actions=0, tokens=0
  ka59-38d34dbb: score=0.

analyzer request failed at action 213: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=29.95281473599971)


[finished] bp35-0a0ad940 state=gave_up level=1/9 score=0.36 actions=212 tokens=38318 per-level=52/21,160/48,0/44,0/38,0/33,0/87,0/86,0/131,0/163 note="tokens=38318"
[finished] m0r0-492f87ba state=gave_up level=0/6 score=0.00 actions=121 tokens=40538 per-level=121/30,0/111,0/203,0/26,0/500,0/237 note="tokens=40538"
[finished] sk48-d8078629 state=gave_up level=0/8 score=0.00 actions=314 tokens=37026 per-level=314/61,0/177,0/101,0/103,0/230,0/181,0/125,0/92 note="tokens=38426"


analyzer request failed at action 38: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=51.95566028999974)


[finished] tn36-ef4dde99 state=gave_up level=0/7 score=0.00 actions=37 tokens=39472 per-level=37/32,0/72,0/26,0/40,0/30,0/55,0/62 note="tokens=40030"
benchmark: regenerated diagnostics in /kaggle/working in 1.60s
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-08-18 10:24:22
ended:     in progress
mean score:    0.01
median score:  0.00
total actions: 799
total tokens:  182276
generated tokens/sec: 100.95 (job wallclock)
total wallclock: 12957.1s

per-game (mean across passes):
  ar25-0c556536: score=0.00, levels=0.0/8, actions=0, tokens=0
  bp35-0a0ad940: score=0.36, levels=1.0/9, actions=212, tokens=38318
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=0, tokens=0
  cn04-2fe56bfb: score=0.00, levels=0.0/6, actions=73, tokens=8620
  dc22-fdcac232: score=0.00, levels=0.0/6, actions=3, tokens=6115
  ft09-0d8bbf25: score=0.00, levels=0.0/6, actions=0, tokens=0
  g50t-5849a774: score=0.00, levels=0.0/7, actions=0,

...

[finished] ft09-0d8bbf25 state=gave_up level=0/6 score=0.00 actions=272 tokens=34535 per-level=272/43,0/12,0/23,0/28,0/65,0/37 note="tokens=38565"


analyzer request failed at action 87: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=100.19571860100041)


[finished] tr87-cd924810 state=gave_up level=0/6 score=0.00 actions=86 tokens=33717 per-level=86/54,0/58,0/40,0/45,0/71,0/146 note="tokens=40015"
benchmark: regenerated diagnostics in /kaggle/working in 1.04s
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-08-18 10:24:22
ended:     in progress
mean score:    0.53
median score:  0.00
total actions: 2857
total tokens:  956261
generated tokens/sec: 99.40 (job wallclock)
total wallclock: 136114.3s

per-game (mean across passes):
  ar25-0c556536: score=0.00, levels=0.0/8, actions=62, tokens=39412
  bp35-0a0ad940: score=0.36, levels=1.0/9, actions=212, tokens=38318
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=69, tokens=40277
  cn04-2fe56bfb: score=0.00, levels=0.0/6, actions=204, tokens=41051
  dc22-fdcac232: score=0.00, levels=0.0/6, actions=46, tokens=38118
  ft09-0d8bbf25: score=0.00, levels=0.0/6, actions=272, tokens=34535
  g50t-5849a774: score=0.00, levels=

analyzer request failed at action 90: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=4.401311025998439)


[finished] g50t-5849a774 state=gave_up level=0/7 score=0.00 actions=89 tokens=51002 per-level=89/78,0/175,0/179,0/230,0/96,0/54,0/67 note="tokens=51002"
benchmark: regenerated diagnostics in /kaggle/working in 3.10s
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-08-18 10:24:22
ended:     2026-08-18 13:19:30
duration:  2h 55m 7s
mean score:    0.53
median score:  0.00
total actions: 2916
total tokens:  986990
generated tokens/sec: 93.93 (job wallclock)
total wallclock: 137036.3s

per-game (mean across passes):
  ar25-0c556536: score=0.00, levels=0.0/8, actions=62, tokens=39412
  bp35-0a0ad940: score=0.36, levels=1.0/9, actions=212, tokens=38318
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=69, tokens=40277
  cn04-2fe56bfb: score=0.00, levels=0.0/6, actions=204, tokens=41051
  dc22-fdcac232: score=0.00, levels=0.0/6, actions=46, tokens=38118
  ft09-0d8bbf25: score=0.00, levels=0.0/6, actions=272, tokens=34535


## 9. Write and validate `submission.parquet`


In [11]:
# === VALIDATED COMPETITION ARTIFACT ===
import pandas as pd

rows = [
    {
        "row_id": f"{run.game_id}_0",
        "game_id": str(run.game_id),
        "end_of_game": _won(run),
        "score": _run_score(run),
    }
    for run in bm.game_runs
]

if len(rows) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Submission requires {RUN_GAME_COUNT} rows; found {len(rows)}"
    )

submission = pd.DataFrame(
    rows,
    columns=["row_id", "game_id", "end_of_game", "score"],
)

if submission["game_id"].astype(str).nunique() != RUN_GAME_COUNT:
    raise RuntimeError("Submission contains duplicate game IDs.")
if submission["score"].isna().any():
    raise RuntimeError("Submission contains missing scores.")

SUBMISSION_PATH = WORKING_DIR / "submission.parquet"
submission.to_parquet(SUBMISSION_PATH, index=False)

check = pd.read_parquet(SUBMISSION_PATH)
if list(check.columns) != ["row_id", "game_id", "end_of_game", "score"]:
    raise RuntimeError(f"Invalid submission columns: {list(check.columns)}")
if len(check) != RUN_GAME_COUNT:
    raise RuntimeError(
        f"Written submission row count mismatch: {len(check)} != {RUN_GAME_COUNT}"
    )

print(
    "SUBMISSION READY "
    f"path={SUBMISSION_PATH} rows={len(check)} "
    f"score_sum={float(check['score'].sum()):.6f}",
    flush=True,
)


SUBMISSION READY path=/kaggle/working/submission.parquet rows=25 score_sum=13.173663


## 10. Final ADL run summary


In [12]:
# === FINAL CLEAN ADL SUMMARY ===
import json
import re

runs = list(bm.game_runs)
scores = [_run_score(run) for run in runs]
levels = [_run_levels(run) for run in runs]
actions = [_run_actions(run) for run in runs]

summary = {
    "schema": "adl.arc3.dual-path.clean.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(runs),
    "mean_score": (sum(scores) / len(scores) if scores else 0.0),
    "positive_score_games": sum(score > 0 for score in scores),
    "total_levels_completed": sum(levels),
    "total_actions": sum(actions),
    "concurrency": TARGET_CONCURRENCY,
    "environment_passes_per_game": 1,
    "internal_candidate_plans_per_decision": 2,
    "second_environment_pass": False,
    "strict_no_prior": True,
    "submission_path": str(SUBMISSION_PATH),
}

summary_path = WORKING_DIR / "dual_path_adl_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("=" * 72)
print("ARC-AGI-3 DUAL-PATH ADL — CLEAN RUN")
print(f"competition_rerun={TRUE_SUBMISSION}")
print("2 internal plans -> 1 selected action -> 1 environment trajectory")
print("SECOND ENVIRONMENT PASS: DISABLED")
print(
    f"games={summary['games']} "
    f"mean_score={summary['mean_score']:.6f} "
    f"positive_games={summary['positive_score_games']} "
    f"levels={summary['total_levels_completed']} "
    f"actions={summary['total_actions']}"
)
print(f"submission={SUBMISSION_PATH}")
print(f"summary={summary_path}")
print("=" * 72)


ARC-AGI-3 DUAL-PATH ADL — CLEAN RUN
competition_rerun=False
2 internal plans -> 1 selected action -> 1 environment trajectory
SECOND ENVIRONMENT PASS: DISABLED
games=25 mean_score=0.526947 positive_games=7 levels=7 actions=2916
submission=/kaggle/working/submission.parquet
summary=/kaggle/working/dual_path_adl_summary.json


## 11. Post-move ADL audit

Checks the current run for a `POST_MOVE_ADL` update after committed moves.


In [13]:
# === POST-MOVE ADL EXTRACTION / AUDIT ===
# Audits this run's artifacts for DUAL_PATH_DECISION and POST_MOVE_ADL markers.
# This does not add prior knowledge; it only summarizes the current execution.

import json
import re
from pathlib import Path

_MARKERS = ("DUAL_PATH_DECISION:", "POST_MOVE_ADL:")
_TEXT_EXTS = {".log", ".txt", ".json", ".jsonl", ".md"}

def _adl_scan_files(root):
    hits = []
    if not root.exists():
        return hits
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in _TEXT_EXTS:
            continue
        if path.name in {
            "dual_path_adl_summary.json",
            "post_move_adl_audit.json",
        }:
            continue
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for marker in _MARKERS:
            count = text.count(marker)
            if count:
                hits.append({
                    "file": str(path),
                    "marker": marker.rstrip(":"),
                    "count": count,
                })
    return hits

_adl_hits = _adl_scan_files(WORKING_DIR)

post_move_count = sum(
    h["count"] for h in _adl_hits if h["marker"] == "POST_MOVE_ADL"
)
decision_count = sum(
    h["count"] for h in _adl_hits if h["marker"] == "DUAL_PATH_DECISION"
)

total_actions = sum(
    len(getattr(run, "history", ()) or ())
    for run in getattr(bm, "game_runs", [])
)

audit = {
    "schema": "adl.arc3.post-move-audit.v1",
    "competition_rerun": bool(TRUE_SUBMISSION),
    "games": len(getattr(bm, "game_runs", []) or []),
    "total_actions": total_actions,
    "dual_path_decision_markers": decision_count,
    "post_move_adl_markers": post_move_count,
    "post_move_coverage": (
        post_move_count / total_actions if total_actions else 0.0
    ),
    "files_with_markers": _adl_hits,
    "required_policy": "POST_MOVE_ADL after every real move before next planning step",
}

audit_path = WORKING_DIR / "post_move_adl_audit.json"
audit_path.write_text(
    json.dumps(audit, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(
    "POST-MOVE ADL AUDIT "
    f"actions={total_actions} "
    f"dual_path_markers={decision_count} "
    f"post_move_markers={post_move_count} "
    f"coverage={audit['post_move_coverage']:.3f}",
    flush=True,
)
print(f"audit={audit_path}", flush=True)


POST-MOVE ADL AUDIT actions=2916 dual_path_markers=2253 post_move_markers=1920 coverage=0.658
audit=/kaggle/working/post_move_adl_audit.json
